In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
parquet_path = "datasets/with_cheaters/134.parquet"

# Load parquet File
df=pd.read_parquet(parquet_path)
original_df=df.copy()

# Load event data
with open('datasets/with_cheaters/134.json', "r") as f:
    events = json.load(f)
events.keys()

dict_keys(['round_poststart', 'player_spawn', 'item_equip', 'smokegrenade_detonate', 'cs_win_panel_match', 'hltv_fixed', 'round_freeze_end', 'bomb_planted', 'hltv_chase', 'round_prestart', 'player_blind', 'cs_round_start_beep', 'player_hurt', 'weapon_zoom', 'buytime_ended', 'bomb_pickup', 'cs_round_final_beep', 'round_officially_ended', 'round_announce_match_start', 'weapon_fire', 'player_jump', 'bullet_damage', 'player_connect', 'round_announce_match_point', 'inferno_startburn', 'server_cvar', 'flashbang_detonate', 'player_footstep', 'bomb_beginplant', 'bomb_dropped', 'player_disconnect', 'cs_pre_restart', 'player_connect_full', 'inferno_expire', 'round_announce_last_round_half', 'begin_new_match', 'announce_phase_end', 'player_death', 'item_pickup', 'rank_update', 'weapon_reload', 'player_team', 'cheaters', 'CSstats_info'])

In [3]:
cheater_ids = {
    player["steamid"]
    for player in events["cheaters"]
}

df["is_cheater"] = (
    df["steamid"].isin(cheater_ids)
).astype("int8")

print(df[["steamid", "is_cheater"]].drop_duplicates())
print(df["total_rounds_played"].value_counts().sort_index())
print("Minimum:", df["total_rounds_played"].min())
print("Maximum:", df["total_rounds_played"].max())
print("Unique rounds:", df["total_rounds_played"].nunique())
print(events.keys())
print(events.get("round_start", [])[:5])
print(events.get("round_end", [])[:5])

     steamid  is_cheater
0  Player_10           0
1   Player_7           1
2   Player_2           1
3   Player_9           1
4   Player_8           1
5   Player_6           1
6   Player_5           0
7   Player_4           0
8   Player_1           0
9   Player_3           1
total_rounds_played
0      47060
1      27360
2      25970
3     102280
4      25380
5      29640
6      23230
7      22000
8      22210
9      23270
10     22960
11     23760
12     31610
13     12620
Name: count, dtype: int64
Minimum: 0
Maximum: 13
Unique rounds: 14
dict_keys(['round_poststart', 'player_spawn', 'item_equip', 'smokegrenade_detonate', 'cs_win_panel_match', 'hltv_fixed', 'round_freeze_end', 'bomb_planted', 'hltv_chase', 'round_prestart', 'player_blind', 'cs_round_start_beep', 'player_hurt', 'weapon_zoom', 'buytime_ended', 'bomb_pickup', 'cs_round_final_beep', 'round_officially_ended', 'round_announce_match_start', 'weapon_fire', 'player_jump', 'bullet_damage', 'player_connect', 'round_announce_match_

In [4]:
df["actual_speed"] = np.sqrt(
    df["velocity_X"]**2 +
    df["velocity_Y"]**2 +
    df["velocity_Z"]**2
)

In [5]:
# Group cumulative player statistics
player_stats = df.groupby("steamid").agg(
    kills=("kills_total", "max"),
    deaths=("deaths_total", "max"),
    headshot_kills=("headshot_kills_total", "max"),
    rounds=("total_rounds_played", "max"), #rounds played till that instance
    avg_speed=("actual_speed", "mean"),
    avg_accuracy_penalty=("accuracy_penalty", "mean")
)

# Basic performance features
player_stats["kills_per_round"] = (
    player_stats["kills"] / player_stats["rounds"].replace(0, np.nan)
)

player_stats["deaths_per_round"] = (
    player_stats["deaths"] / player_stats["rounds"].replace(0, np.nan)
)

player_stats["kd_ratio"] = (
    player_stats["kills"] / player_stats["deaths"].replace(0, np.nan)
)

player_stats["headshot_rate"] = (
    player_stats["headshot_kills"] /
    player_stats["kills"].replace(0, np.nan) * 100
)

# Calculate total damage dealt by each player
damage = {}

for event in events["player_hurt"]:
    attacker = event["attacker_steamid"]
    damage[attacker] = (
        damage.get(attacker, 0) + event["dmg_health"]
    )

player_stats["total_damage"] = (
    pd.Series(damage).reindex(player_stats.index).fillna(0)
)


# Damage per round
player_stats["damage_per_round"] = (
    player_stats["total_damage"] /
    player_stats["rounds"].replace(0, np.nan)
)
player_stats["is_cheater"] = (
    df["steamid"].isin(cheater_ids)
).astype("int8")
print(player_stats)

           kills  deaths  headshot_kills  rounds   avg_speed  \
steamid                                                        
Player_1       0      13               0      13   14.522911   
Player_10      1       2               1      13   74.734695   
Player_2       1      13               1      13   33.427681   
Player_3       1       6               1      13   53.727524   
Player_4       0       5               0      13   41.591747   
Player_5       0      13               0      13   22.418692   
Player_6       5       3               4      13   98.922920   
Player_7       6       1               4      13  136.411438   
Player_8      30       1              21      13  167.384720   
Player_9       1       1               1      13   86.512558   

           avg_accuracy_penalty  kills_per_round  deaths_per_round   kd_ratio  \
steamid                                                                         
Player_1               0.007738         0.000000          1.000000   

In [6]:
# Create a DataFrame from player death events
death_events = pd.DataFrame(events["player_death"])

# Group by attacker
combat_stats = death_events.groupby("attacker_steamid").agg(
    total_kills=("attacker_steamid", "count"),
    smoke_kills=("thrusmoke", "sum"),
    blind_kills=("attackerblind", "sum"),
    air_kills=("attackerinair", "sum")
)

# Calculate ratios
combat_stats["smoke_kill_ratio"] = (
    combat_stats["smoke_kills"] / combat_stats["total_kills"]
)

combat_stats["blind_kill_ratio"] = (
    combat_stats["blind_kills"] / combat_stats["total_kills"]
)

combat_stats["air_kill_ratio"] = (
    combat_stats["air_kills"] / combat_stats["total_kills"]
)

# Merge with existing player statistics
player_stats = player_stats.join(
    combat_stats.drop(columns=["total_kills"]),
    how="left"
)

# Players with no kills receive zero for these features
player_stats[
    ["smoke_kill_ratio", "blind_kill_ratio", "air_kill_ratio"]
] = player_stats[
    ["smoke_kill_ratio", "blind_kill_ratio", "air_kill_ratio"]
].fillna(0)

player_stats["has_kills"] = (
    player_stats["kills"] > 0
).astype(int)

player_stats["headshot_rate"] = (
    player_stats["headshot_rate"].fillna(0)
)

event_features = [
    "smoke_kills",
    "blind_kills",
    "air_kills",
    "smoke_kill_ratio",
    "blind_kill_ratio",
    "air_kill_ratio"
]

player_stats[event_features] = (
    player_stats[event_features].fillna(0)
)
match_id = Path(parquet_path).stem
player_stats["match_id"] = match_id
print(player_stats)

           kills  deaths  headshot_kills  rounds   avg_speed  \
steamid                                                        
Player_1       0      13               0      13   14.522911   
Player_10      1       2               1      13   74.734695   
Player_2       1      13               1      13   33.427681   
Player_3       1       6               1      13   53.727524   
Player_4       0       5               0      13   41.591747   
Player_5       0      13               0      13   22.418692   
Player_6       5       3               4      13   98.922920   
Player_7       6       1               4      13  136.411438   
Player_8      30       1              21      13  167.384720   
Player_9       1       1               1      13   86.512558   

           avg_accuracy_penalty  kills_per_round  deaths_per_round   kd_ratio  \
steamid                                                                         
Player_1               0.007738         0.000000          1.000000   

In [7]:
features=[
"kills_per_round",
"deaths_per_round",
"kd_ratio",
"headshot_rate",
"has_kills",
"avg_speed",
"avg_accuracy_penalty",
"damage_per_round", 
"smoke_kill_ratio",	
"blind_kill_ratio",
"air_kill_ratio"
]
#include below features with more data later


X = player_stats[features]

y = player_stats['is_cheater']

In [8]:
X

,kills_per_round,deaths_per_round,kd_ratio,headshot_rate,has_kills,avg_speed,avg_accuracy_penalty,damage_per_round,smoke_kill_ratio,blind_kill_ratio,air_kill_ratio
steamid,,,,,,,,,,,
Player_1,0.000000,1.000000,0.000000,0.000000,0,14.522911,0.007738,15.384615,0.000000,0.0,0.000000
Player_10,0.076923,0.153846,0.500000,100.000000,1,74.734695,0.008081,25.384615,0.000000,0.0,0.000000
Player_2,0.076923,1.000000,0.076923,100.000000,1,33.427681,0.006306,39.000000,0.000000,0.0,0.000000
Player_3,0.076923,0.461538,0.166667,100.000000,1,53.727524,0.014357,8.692308,0.000000,0.0,0.000000
Player_4,0.000000,0.384615,0.000000,0.000000,0,41.591747,0.002139,2.615385,0.000000,0.0,0.000000
Player_5,0.000000,1.000000,0.000000,0.000000,0,22.418692,0.004024,7.076923,0.000000,0.0,0.000000
Player_6,0.384615,0.230769,1.666667,80.000000,1,98.922920,0.011345,54.692308,0.000000,0.0,0.666667
Player_7,0.461538,0.076923,6.000000,66.666667,1,136.411438,0.010389,76.461538,0.000000,0.0,0.000000
Player_8,2.307692,0.076923,30.000000,70.000000,1,167.384720,0.015791,343.692308,0.064516,0.0,0.096774


In [9]:
y

steamid
Player_1    NaN
Player_10   NaN
Player_2    NaN
Player_3    NaN
Player_4    NaN
Player_5    NaN
Player_6    NaN
Player_7    NaN
Player_8    NaN
Player_9    NaN
Name: is_cheater, dtype: float64

In [10]:
all_matches = pd.read_csv('combined_player_stats.csv')
print(all_matches["is_cheater"].value_counts())
print(all_matches["is_cheater"].value_counts(normalize=True))
print(
    all_matches.groupby("match_id")["is_cheater"].sum()
)

is_cheater
0    432
1    118
Name: count, dtype: int64
is_cheater
0    0.785455
1    0.214545
Name: proportion, dtype: float64
match_id
0      0
12     0
100    1
107    4
108    5
109    5
110    4
111    5
112    4
113    3
114    3
115    5
116    5
117    8
118    0
119    0
120    0
121    0
122    0
123    0
124    0
125    0
126    0
129    0
130    3
131    3
132    1
133    3
134    6
135    6
136    4
137    7
138    5
139    4
140    6
141    4
142    3
143    3
144    3
145    5
146    0
147    0
148    0
149    0
150    0
151    0
152    0
153    0
154    0
155    0
156    0
157    0
158    0
159    0
160    0
Name: is_cheater, dtype: int64


In [11]:
duplicates = all_matches.duplicated(
    subset=["match_id", "steamid"]
)

print("Duplicate records:", duplicates.sum())

Duplicate records: 0
